In [ ]:
import subprocess
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from scipy.stats import rankdata
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

In [ ]:
KAGGLE_DIR = Path("/kaggle/input/playground-series-s6e11")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path("data")

try:
    USE_GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    USE_GPU = False

# Trên CPU cá nhân: CatBoost rất chậm nên mặc định chỉ chạy XGB + LGB (~4-5 phút).
# Lên Kaggle bật GPU sẽ tự động bật cả CatBoost (hoặc đổi thành True nếu bạn muốn đợi trên CPU).
INCLUDE_CATBOOST = USE_GPU
print(f"DATA_DIR: {DATA_DIR} | USE_GPU: {USE_GPU} | INCLUDE_CATBOOST: {INCLUDE_CATBOOST}")

BOOL_COLS = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]

CAT_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Chuyển kiểu boolean sang 0/1, giữ nguyên 24 cột gốc để tránh nhiễu cây quyết định."""
    out = df.copy()
    out[BOOL_COLS] = out[BOOL_COLS].astype(int)
    return out


def add_fold_group_features(
    X_tr: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Tính đặc trưng tổng hợp theo nhóm từ X_tr để tránh rò rỉ dữ liệu."""
    tr, val, te = X_tr.copy(), X_val.copy(), X_test.copy()

    cat_amt_mean = tr.groupby("merchant_category", observed=False)["amount_usd"].mean()
    ch_amt_mean = tr.groupby("channel", observed=False)["amount_usd"].mean()
    cat_risk_mean = tr.groupby("merchant_category", observed=False)["merchant_risk_score"].mean()

    for part in (tr, val, te):
        part["amount_ratio_cat"] = part["amount_usd"] / (part["merchant_category"].map(cat_amt_mean).astype(float) + 1.0)
        part["amount_ratio_ch"] = part["amount_usd"] / (part["channel"].map(ch_amt_mean).astype(float) + 1.0)
        part["risk_diff_cat"] = part["merchant_risk_score"] - part["merchant_category"].map(cat_risk_mean).astype(float)

    return tr, val, te

In [ ]:
LR = 0.03 if USE_GPU else 0.05
N_EST = 3000 if USE_GPU else 2000
EARLY_STOP = 80 if USE_GPU else 50

def build_xgb() -> XGBClassifier:
    return XGBClassifier(
        n_estimators=N_EST,
        learning_rate=LR,
        max_depth=5,
        min_child_weight=30,
        subsample=0.8,
        colsample_bytree=0.75,
        reg_alpha=2.0,
        reg_lambda=10.0,
        tree_method="hist",
        device="cuda" if USE_GPU else "cpu",
        enable_categorical=True,
        eval_metric="auc",
        early_stopping_rounds=EARLY_STOP,
        n_jobs=-1,
        random_state=42,
    )


def build_lgb() -> LGBMClassifier:
    return LGBMClassifier(
        n_estimators=N_EST,
        learning_rate=LR,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.75,
        reg_alpha=2.0,
        reg_lambda=10.0,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )


def build_cat() -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=N_EST,
        learning_rate=0.04 if USE_GPU else 0.08,
        depth=6,
        l2_leaf_reg=10.0,
        max_ctr_complexity=4 if USE_GPU else 1,
        eval_metric="AUC",
        task_type="GPU" if USE_GPU else "CPU",
        thread_count=-1,
        random_seed=42,
        verbose=0,
    )

In [ ]:
# 1. Đọc dữ liệu & đồng bộ category
train = add_features(pd.read_csv(DATA_DIR / "train.csv"))
test = add_features(pd.read_csv(DATA_DIR / "test.csv"))

for col in CAT_COLS:
    cats = pd.CategoricalDtype(categories=sorted(set(train[col].dropna()).union(test[col].dropna())))
    train[col] = train[col].astype(cats)
    test[col] = test[col].astype(cats)

X = train.drop(columns=["id", "is_fraud"])
y = train["is_fraud"].astype(int)
X_test = test.drop(columns=["id"])

# 2. Stratified 5-Fold Cross-Validation cho cả 3 mô hình
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_xgb, test_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_lgb, test_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat, test_cat = np.zeros(len(X)), np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    t0 = time.time()
    X_tr, X_val, X_te_fold = add_fold_group_features(
        X.iloc[tr_idx], X.iloc[val_idx], X_test
    )
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # --- XGBoost ---
    xgb_model = build_xgb()
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb += xgb_model.predict_proba(X_te_fold)[:, 1] / n_splits

    # --- LightGBM ---
    lgb_model = build_lgb()
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)],
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val)[:, 1]
    test_lgb += lgb_model.predict_proba(X_te_fold)[:, 1] / n_splits

    msg = (
        f"Fold {fold}/{n_splits} ({time.time() - t0:.1f}s) | "
        f"XGB={roc_auc_score(y_val, oof_xgb[val_idx]):.5f} | "
        f"LGB={roc_auc_score(y_val, oof_lgb[val_idx]):.5f}"
    )

    # --- CatBoost (tự chạy trên GPU Kaggle hoặc khi INCLUDE_CATBOOST=True) ---
    if INCLUDE_CATBOOST:
        cat_model = build_cat()
        cat_model.fit(X_tr, y_tr, cat_features=CAT_COLS, eval_set=(X_val, y_val), early_stopping_rounds=EARLY_STOP)
        oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
        test_cat += cat_model.predict_proba(X_te_fold)[:, 1] / n_splits
        msg += f" | CAT={roc_auc_score(y_val, oof_cat[val_idx]):.5f}"

    print(msg)

In [ ]:
# 3. Đánh giá từng mô hình & tìm trọng số Ensemble tối ưu trên OOF
print(f"OOF XGBoost : {roc_auc_score(y, oof_xgb):.5f}")
print(f"OOF LightGBM: {roc_auc_score(y, oof_lgb):.5f}")
if INCLUDE_CATBOOST:
    print(f"OOF CatBoost: {roc_auc_score(y, oof_cat):.5f}")

# Lưu lại dự đoán phòng khi cần thử nghiệm thêm
np.save("oof_xgb.npy", oof_xgb); np.save("test_xgb.npy", test_xgb)
np.save("oof_lgb.npy", oof_lgb); np.save("test_lgb.npy", test_lgb)
if INCLUDE_CATBOOST:
    np.save("oof_cat.npy", oof_cat); np.save("test_cat.npy", test_cat)

r_xgb = rankdata(oof_xgb) / len(oof_xgb)
r_lgb = rankdata(oof_lgb) / len(oof_lgb)
r_cat = (rankdata(oof_cat) / len(oof_cat)) if INCLUDE_CATBOOST else np.zeros(len(oof_xgb))

def eval_grid(candidates: list[tuple[float, float, float]]) -> tuple[float, tuple[float, float, float], str]:
    top_auc, top_w, top_mode = -1.0, (0.5, 0.5, 0.0), "prob"
    for w1, w2, w3 in candidates:
        auc_p = roc_auc_score(y, w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cat)
        if auc_p > top_auc:
            top_auc, top_w, top_mode = auc_p, (w1, w2, w3), "prob"
        auc_r = roc_auc_score(y, w1 * r_xgb + w2 * r_lgb + w3 * r_cat)
        if auc_r > top_auc:
            top_auc, top_w, top_mode = auc_r, (w1, w2, w3), "rank"
    return top_auc, top_w, top_mode

if INCLUDE_CATBOOST:
    coarse = [
        (w1, w2, round(1.0 - w1 - w2, 2))
        for w1 in np.round(np.linspace(0, 1, 11), 2)
        for w2 in np.round(np.linspace(0, 1, 11), 2)
        if w1 + w2 <= 1.0001
    ]
    _, (cw1, cw2, _), _ = eval_grid(coarse)
    candidates = [
        (round(w1, 2), round(w2, 2), round(1.0 - w1 - w2, 2))
        for w1 in np.arange(max(0.0, cw1 - 0.1), min(1.0, cw1 + 0.101), 0.02)
        for w2 in np.arange(max(0.0, cw2 - 0.1), min(1.0, cw2 + 0.101), 0.02)
        if 0.0 <= 1.0 - w1 - w2 <= 1.0001
    ]
else:
    candidates = [(round(w1, 2), round(1.0 - w1, 2), 0.0) for w1 in np.round(np.linspace(0, 1, 21), 2)]

best_auc, best_weights, best_mode = eval_grid(candidates)

w1, w2, w3 = best_weights
print(f"\nBest Ensemble ({best_mode}) weights (XGB, LGB, CAT) = ({w1}, {w2}, {w3})")
print(f"Overall OOF Ensemble ROC-AUC: {best_auc:.5f}")

oof_ens = w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cat
print(classification_report(y, (oof_ens >= 0.5).astype(int)))

if best_mode == "prob":
    test_ens = w1 * test_xgb + w2 * test_lgb + w3 * test_cat
else:
    test_ens = (
        w1 * (rankdata(test_xgb) / len(test_xgb))
        + w2 * (rankdata(test_lgb) / len(test_lgb))
        + w3 * (rankdata(test_cat) / len(test_cat))
    )

submission = pd.DataFrame({
    "id": test["id"],
    "is_fraud": test_ens,
})
submission.to_csv("submission.csv", index=False)
print("Đã lưu submission.csv:", submission.shape)